In [ ]:
#| default_exp table_and_charts

# tables and charts
> Load Excel from `/chat/send` into pandas, then render Plotly table/chart panels. Table mode can forward questions to the data analyst agent.

In [ ]:
#| export
import base64
from dataclasses import dataclass, field
from datetime import datetime, timezone
from io import BytesIO
from typing import Any

import pandas as pd
import plotly.graph_objects as go
from fasthtml.common import *
from monsterui.all import *

## excel → pandas → plotly table

In [ ]:
#| export
def excel_to_df(raw: bytes) -> pd.DataFrame:
    """Load Excel bytes into a pandas DataFrame."""
    return pd.read_excel(BytesIO(raw))


def df_to_plotly_table(df: pd.DataFrame) -> go.Figure:
    """Build a Plotly table figure from a DataFrame."""
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(df.columns),
            fill_color="paleturquoise",
            align="left",
        ),
        cells=dict(
            values=[df[col] for col in df.columns],
            fill_color="lavender",
            align="left",
        ),
    )])
    fig.update_layout(
        autosize=True,
        margin=dict(l=8, r=8, t=8, b=8),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )
    return fig

## Store — local table + analyst id + chart

In [ ]:
#| export
@dataclass
class TableResult:
    """Latest Excel result as a DataFrame + Plotly figure."""
    df: pd.DataFrame
    fig: go.Figure
    title: str = "Query results"
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))

    @property
    def n_rows(self) -> int:
        return len(self.df)


_latest_table: TableResult | None = None
_analyst_table_id: str | None = None
_latest_chart: go.Figure | None = None
_latest_chart_title: str = "Chart"


def get_latest_table() -> TableResult | None:
    return _latest_table


def get_analyst_table_id() -> str | None:
    return _analyst_table_id


def set_analyst_table_id(table_id: str | None) -> str | None:
    """Remember the remote data-analyst store id for the current local table."""
    global _analyst_table_id
    _analyst_table_id = (table_id or "").strip() or None
    return _analyst_table_id


def get_latest_chart() -> go.Figure | None:
    return _latest_chart


def store_chart_from_plotly_json(
    plotly_json: dict[str, Any] | None,
    *,
    title: str | None = None,
) -> go.Figure | None:
    """Persist a Plotly figure built from the analyst's ``plotly_json`` payload."""
    global _latest_chart, _latest_chart_title
    if not plotly_json:
        return _latest_chart
    fig = go.Figure(plotly_json)
    _latest_chart = fig
    if title:
        _latest_chart_title = title
    return _latest_chart


def store_table_result(result: TableResult | None) -> TableResult | None:
    """Store a new local table and clear analyst/chart bindings (new data)."""
    global _latest_table, _analyst_table_id, _latest_chart
    _latest_table = result
    _analyst_table_id = None
    _latest_chart = None
    return _latest_table


def _excel_bytes_from_blob(blob: dict) -> tuple[str, bytes] | None:
    fname = blob.get("filename") or blob.get("name") or "results.xlsx"
    for key in ("content", "data", "base64", "file", "file_content", "content_base64", "bytes"):
        val = blob.get(key)
        if isinstance(val, str) and val.strip():
            try:
                return str(fname), base64.b64decode(val)
            except (ValueError, TypeError):
                continue
    return None


def extract_excel_from_parsed(parsed: Any) -> tuple[str, bytes] | None:
    """Find Excel bytes in controller ``parsed`` (top-level or nested under ``excel``)."""
    if not isinstance(parsed, dict):
        return None
    blob = parsed
    fmt = (blob.get("format") or "").lower()
    if fmt != "excel_file":
        nested = blob.get("excel")
        if isinstance(nested, dict):
            blob = nested
        else:
            return None
    return _excel_bytes_from_blob(blob)


def ingest_parsed(parsed: Any) -> TableResult | None:
    """Excel from ``parsed`` → pandas → Plotly table, then store as latest result."""
    extracted = extract_excel_from_parsed(parsed)
    if not extracted:
        return None
    fname, raw = extracted
    df = excel_to_df(raw)
    fig = df_to_plotly_table(df)
    return store_table_result(TableResult(df=df, fig=fig, title=fname))


def df_records_for_analyst(df: pd.DataFrame) -> list[dict[str, Any]]:
    """JSON-safe row dicts for ``POST /tables`` on the data analyst."""
    clean = df.astype(object).where(pd.notna(df), None)
    return clean.to_dict(orient="records")

## UI — table, chart, analyze reply

In [ ]:
#| export
def table_badge(*, oob: bool = False):
    result = get_latest_table()
    n = result.n_rows if result else 0
    attrs: dict = {"id": "table-badge"}
    if oob:
        attrs["hx_swap_oob"] = "true"
    cls = "badge badge-xs badge-primary absolute -top-1 -right-1"
    if not n:
        cls += " hidden"
    return Span(str(n) if n else "", **attrs, cls=cls)


def chart_badge(*, oob: bool = False):
    has = get_latest_chart() is not None
    attrs: dict = {"id": "chart-badge"}
    if oob:
        attrs["hx_swap_oob"] = "true"
    cls = "badge badge-xs badge-primary absolute -top-1 -right-1"
    if not has:
        cls += " hidden"
    return Span("1" if has else "", **attrs, cls=cls)


_PLOTLY_RESIZE_JS_TABLE = """
(function () {
  const el = document.querySelector('#table-panel .js-plotly-plot');
  if (el && window.Plotly && Plotly.Plots) Plotly.Plots.resize(el);
})();
"""

_PLOTLY_RESIZE_JS_CHART = """
(function () {
  const el = document.querySelector('#chart-panel .js-plotly-plot');
  if (el && window.Plotly && Plotly.Plots) Plotly.Plots.resize(el);
})();
"""


def table_panel(*, oob: bool = False):
    """Show the latest Plotly table filling the available viewport height."""
    result = get_latest_table()
    panel_kw: dict = {
        "id": "table-panel",
        "cls": "flex flex-col flex-1 min-h-0 h-full overflow-hidden",
    }
    if oob:
        panel_kw["hx_swap_oob"] = "true"

    if not result or result.df.empty:
        return Div(
            Div(
                UkIcon("table-2", cls="w-12 h-12 text-base-content/20"),
                P("No table data yet", cls="text-base-content/50 mt-3"),
                P("Ask a data question in chat and results will appear here.",
                  cls="text-sm text-base-content/30 mt-1 text-center max-w-xs"),
                cls="flex flex-col items-center justify-center flex-1 py-16",
            ),
            **panel_kw,
        )

    plot_html = result.fig.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config={"responsive": True, "displayModeBar": False},
        default_width="100%",
        default_height="100%",
    )
    return Div(
        Div(
            H3(result.title, cls="text-lg font-semibold mb-1"),
            P(
                f"{result.n_rows} row{'s' if result.n_rows != 1 else ''} · {result.created_at}",
                cls="text-sm text-base-content/50",
            ),
            cls="px-4 pt-4 pb-2 shrink-0",
        ),
        Div(
            NotStr(plot_html),
            cls="w-full px-2 pb-2",
            style="flex:1 1 auto; min-height:0; height:calc(100vh - 14rem);",
        ),
        Script(_PLOTLY_RESIZE_JS_TABLE),
        **panel_kw,
    )


def analyze_reply(text: str = "", *, oob: bool = False):
    """Banner above the table for data-analyst answers while Table mode is active."""
    attrs: dict = {"id": "analyze-reply"}
    if oob:
        attrs["hx_swap_oob"] = "true"
    if not (text or "").strip():
        return Div(**attrs, cls="hidden")
    return Div(
        Div(
            P("Analyst", cls="text-xs font-semibold uppercase tracking-wide text-primary mb-1"),
            P(text, cls="text-sm whitespace-pre-line"),
            cls="mx-4 mt-3 mb-1 p-3 rounded-lg bg-base-200 border border-base-300",
        ),
        **attrs,
    )


def table_view():
    return Div(
        analyze_reply(""),
        table_panel(),
        id="table-view",
        cls="flex flex-col flex-1 min-h-0 h-full hidden",
    )


def chart_panel(*, oob: bool = False):
    """Show the latest analyst chart under the Chart icon."""
    fig = get_latest_chart()
    panel_kw: dict = {
        "id": "chart-panel",
        "cls": "flex flex-col flex-1 min-h-0 h-full overflow-hidden",
    }
    if oob:
        panel_kw["hx_swap_oob"] = "true"

    if fig is None:
        return Div(
            Div(
                UkIcon("bar-chart-2", cls="w-12 h-12 text-base-content/20"),
                P("No chart yet", cls="text-base-content/50 mt-3"),
                P("With the Table icon active, ask for a chart about the loaded data.",
                  cls="text-sm text-base-content/30 mt-1 text-center max-w-xs"),
                cls="flex flex-col items-center justify-center flex-1 py-16",
            ),
            **panel_kw,
        )

    plot_html = fig.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config={"responsive": True, "displayModeBar": False},
        default_width="100%",
        default_height="100%",
    )
    return Div(
        Div(
            H3(_latest_chart_title, cls="text-lg font-semibold mb-1"),
            cls="px-4 pt-4 pb-2 shrink-0",
        ),
        Div(
            NotStr(plot_html),
            cls="w-full px-2 pb-2",
            style="flex:1 1 auto; min-height:0; height:calc(100vh - 11.5rem);",
        ),
        Script(_PLOTLY_RESIZE_JS_CHART),
        **panel_kw,
    )


def chart_view():
    return Div(
        chart_panel(),
        id="chart-view",
        cls="flex flex-col flex-1 min-h-0 h-full hidden",
    )


def register_table_routes(rt):
    @rt("/view/table")
    def view_table():
        return table_panel()

    @rt("/view/chart")
    def view_chart():
        return chart_panel()

    return view_table

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()